# Day 062 | Intro to PyTorch — Tensors

## 100 Days of ML for Geotechnical Engineering
**Phase:** Deep Learning | **Author:** Ripon Chandra Malo | **Date:** YYYY-MM-DD

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| Tensor | Multi-dimensional array (like numpy) on CPU/GPU | Soil property matrix across depth |
| Tensor creation | Various ways to build tensors | Creating depth, unit weight, stress profiles |
| Tensor operations | Math on tensors | Computing effective stress from total stress - pore pressure |
| Tensor <-> NumPy | Seamless conversion | Importing lab data as numpy, converting for PyTorch |
| GPU concept | Parallel computation on graphics cards | Processing thousands of soil samples simultaneously |
| dtype and shape | Data type and dimensions | Ensuring stress values are float32 for training |

---

## Our Geotechnical Problem

We will use PyTorch tensors to compute an **overburden stress profile** through a multi-layer soil deposit:

```
Surface ──────────────────────
  Layer 1: Fill (gamma = 18 kN/m3, 2m thick)
──────────────────────────────
  Layer 2: Soft Clay (gamma = 16 kN/m3, 4m thick)
──────────────────────────────
  Layer 3: Sand (gamma = 19 kN/m3, 3m thick)
──────────────────────────────
  Layer 4: Stiff Clay (gamma = 20 kN/m3, 6m thick)
──────────────────────────────
         depth

Total vertical stress:  sigma_v = sum(gamma_i * h_i)
Pore water pressure:    u = gamma_w * (z - z_wt)
Effective stress:       sigma'_v = sigma_v - u
```

We will perform all calculations using **PyTorch tensors** instead of NumPy arrays.

---

## Part 1: Installing and Importing PyTorch

PyTorch is the leading deep learning framework. Tensors are its fundamental data structure — like NumPy arrays but with GPU support and automatic differentiation.

In [ ]:
# ─────────────────────────────────────────────────────────
# IMPORT PYTORCH AND CHECK VERSION
# ─────────────────────────────────────────────────────────

import torch
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device:      {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU — that is fine for learning!")
print(f"\nDefault dtype:   {torch.get_default_dtype()}")

---

## Part 2: Creating Tensors — Multiple Methods

There are many ways to create tensors. Each has a specific use case in geotechnical modeling.

In [ ]:
# ─────────────────────────────────────────────────────────
# TENSOR CREATION — From soil data
# ─────────────────────────────────────────────────────────

# Method 1: From a Python list — soil unit weights (kN/m3)
gamma = torch.tensor([18.0, 16.0, 19.0, 20.0])
print(f"Unit weights: {gamma}")
print(f"  Shape: {gamma.shape}, dtype: {gamma.dtype}")

# Method 2: From a list of lists — SPT N-values at 3 boreholes x 5 depths
spt_data = torch.tensor([
    [5, 8, 12, 18, 25],    # Borehole 1
    [4, 7, 10, 15, 22],    # Borehole 2
    [6, 9, 14, 20, 28],    # Borehole 3
], dtype=torch.float32)
print(f"\nSPT data: {spt_data.shape}  (3 boreholes x 5 depths)")

# Method 3: Zeros — initialize settlement array
settlement = torch.zeros(10)  # 10 monitoring points
print(f"\nSettlement (init): {settlement}")

# Method 4: Ones — initial safety factors
fs = torch.ones(5) * 1.5  # factor of safety = 1.5
print(f"Safety factors:    {fs}")

# Method 5: Arange — depth increments
depths = torch.arange(0, 15.5, 0.5)  # 0 to 15m, 0.5m increments
print(f"\nDepth profile: {depths[:10]}... ({len(depths)} points)")

# Method 6: Linspace — evenly spaced depths
z = torch.linspace(0, 15, 31)  # 31 points from 0 to 15m
print(f"Linspace depths: {z[:6]}... ({len(z)} points)")

# Method 7: Random — synthetic soil properties
torch.manual_seed(42)
cohesion = torch.rand(100) * 50 + 10  # uniform [10, 60] kPa
print(f"\nRandom cohesion values: shape={cohesion.shape}, range=[{cohesion.min():.1f}, {cohesion.max():.1f}] kPa")

In [ ]:
# ─────────────────────────────────────────────────────────
# TENSOR ATTRIBUTES — Shape, dtype, device
# ─────────────────────────────────────────────────────────

# Create a 2D tensor: soil properties for 5 samples
# Columns: [N_value, depth(m), water_content(%), unit_weight(kN/m3)]
soil_props = torch.tensor([
    [10, 2.0, 35, 16.5],
    [15, 4.5, 28, 17.2],
    [22, 7.0, 22, 18.1],
    [30, 10.0, 18, 19.0],
    [38, 13.5, 15, 19.8],
])

print("Soil properties tensor:")
print(f"  Shape:       {soil_props.shape}   (5 samples x 4 features)")
print(f"  Dimensions:  {soil_props.ndim}D")
print(f"  dtype:       {soil_props.dtype}")
print(f"  device:      {soil_props.device}")
print(f"  # elements:  {soil_props.numel()}")
print(f"  Memory:      {soil_props.element_size() * soil_props.numel()} bytes")
print()

# Casting dtype
soil_float32 = soil_props.float()  # shorthand for .to(torch.float32)
print(f"After .float():  dtype = {soil_float32.dtype}")

soil_float64 = soil_props.double()  # torch.float64
print(f"After .double(): dtype = {soil_float64.dtype}")

---

## Part 3: Tensor Operations — Geotechnical Calculations

PyTorch tensors support all standard math operations, element-wise and matrix operations.

In [ ]:
# ─────────────────────────────────────────────────────────
# ELEMENT-WISE OPERATIONS — Stress calculations
# ─────────────────────────────────────────────────────────

# Total stress at different depths (sigma_v = gamma * z)
gamma_soil = torch.tensor(18.5)   # kN/m3
depths = torch.tensor([1.0, 2.0, 3.0, 5.0, 8.0, 10.0, 15.0])  # meters

# Element-wise multiplication (broadcasting)
sigma_v = gamma_soil * depths
print("Total vertical stress profile:")
print(f"  Depths (m):       {depths.tolist()}")
print(f"  sigma_v (kPa):    {sigma_v.tolist()}")

# Pore water pressure (water table at 2m depth)
gamma_w = torch.tensor(9.81)  # kN/m3
z_wt = 2.0  # water table depth
u = gamma_w * torch.clamp(depths - z_wt, min=0)  # clamp: no negative pore pressure
print(f"\n  Pore pressure (kPa): {u.tolist()}")

# Effective stress
sigma_eff = sigma_v - u
print(f"  sigma'_v (kPa):      {[f'{s:.2f}' for s in sigma_eff.tolist()]}")

# Arithmetic operations
print(f"\nMax effective stress: {sigma_eff.max():.2f} kPa")
print(f"Mean effective stress: {sigma_eff.mean():.2f} kPa")
print(f"Sum of depths: {depths.sum():.1f} m")

In [ ]:
# ─────────────────────────────────────────────────────────
# MATRIX OPERATIONS — Combining soil layer properties
# ─────────────────────────────────────────────────────────

# Matrix multiplication example:
# Weight matrix (3 outputs from 4 features) — like a linear layer
torch.manual_seed(42)
features = torch.tensor([[10.0, 2.0, 35.0, 16.5],
                          [22.0, 7.0, 22.0, 18.1]])  # 2 samples x 4 features

weights = torch.randn(4, 3)  # 4 features -> 3 outputs

# Matrix multiply: (2x4) @ (4x3) -> (2x3)
output = features @ weights  # or torch.matmul(features, weights)
print(f"Features shape:  {features.shape}")
print(f"Weights shape:   {weights.shape}")
print(f"Output shape:    {output.shape}")
print(f"Output:\n{output}")

# Transpose
print(f"\nTranspose of features: {features.T.shape}")

# Dot product — used in similarity calculations
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])
print(f"\nDot product: {torch.dot(a, b):.1f}")

---

## Part 4: Indexing and Slicing — Extracting Soil Data

Tensor indexing works just like NumPy — essential for selecting specific soil layers or properties.

In [ ]:
# ─────────────────────────────────────────────────────────
# INDEXING AND SLICING TENSORS
# ─────────────────────────────────────────────────────────

# Create CPT data: 8 depth readings x 3 measurements (qc, fs, u2)
cpt_data = torch.tensor([
    [0.5, 0.02, 10],    # depth 1m
    [1.2, 0.04, 25],    # depth 2m
    [0.8, 0.05, 40],    # depth 3m
    [3.5, 0.08, 15],    # depth 4m
    [5.2, 0.12, 20],    # depth 5m
    [8.0, 0.15, 30],    # depth 6m
    [12.0, 0.20, 10],   # depth 7m
    [15.5, 0.25, 5],    # depth 8m
])

print(f"CPT data shape: {cpt_data.shape}  (8 depths x 3 measurements)")

# Select single element
print(f"\nqc at depth 5m: {cpt_data[4, 0]:.1f} MPa")

# Select a row (all measurements at one depth)
print(f"All data at depth 3m: {cpt_data[2]}")

# Select a column (one measurement at all depths)
qc_values = cpt_data[:, 0]  # all rows, column 0
print(f"Cone resistance profile: {qc_values}")

# Slice rows 3-6 (depths 4-7m)
deep_data = cpt_data[3:7, :]
print(f"\nData from 4-7m depth:\n{deep_data}")

# Boolean indexing — find layers where qc > 5 MPa (dense layers)
dense_mask = cpt_data[:, 0] > 5.0
print(f"\nDense layers (qc > 5 MPa):")
print(f"  Mask: {dense_mask}")
print(f"  Data:\n{cpt_data[dense_mask]}")

---

## Part 5: Reshaping Tensors

Neural networks require specific input shapes. Reshaping is critical when preparing geotechnical data for models.

In [ ]:
# ─────────────────────────────────────────────────────────
# RESHAPING — Preparing data for neural networks
# ─────────────────────────────────────────────────────────

# Flatten a 2D tensor to 1D
flat = cpt_data.flatten()
print(f"Flattened: {cpt_data.shape} -> {flat.shape}")

# Reshape: 1D -> 2D
reshaped = flat.reshape(8, 3)
print(f"Reshaped back: {flat.shape} -> {reshaped.shape}")

# View (shares memory — more efficient than reshape)
viewed = flat.view(8, 3)
print(f"View: {viewed.shape}  (shares memory with original)")

# Unsqueeze — add a dimension (common for batch dimension)
single_sample = torch.tensor([10.0, 5.0, 25.0])  # (3,)
batched = single_sample.unsqueeze(0)               # (1, 3) — add batch dim
print(f"\nSingle sample: {single_sample.shape} -> unsqueeze(0) -> {batched.shape}")

# Squeeze — remove dimensions of size 1
output = torch.tensor([[[42.0]]])  # (1, 1, 1)
squeezed = output.squeeze()         # scalar
print(f"Squeeze: {output.shape} -> {squeezed.shape}  value = {squeezed.item():.1f}")

# Stack — combine multiple 1D profiles into a batch
profile_1 = torch.tensor([10.0, 20.0, 35.0, 50.0])  # stress at 4 depths
profile_2 = torch.tensor([12.0, 22.0, 38.0, 55.0])
profile_3 = torch.tensor([8.0, 18.0, 30.0, 45.0])

batch = torch.stack([profile_1, profile_2, profile_3])  # (3, 4)
print(f"\nStacked profiles: {batch.shape}  (3 boreholes x 4 depths)")

---

## Part 6: Tensor <-> NumPy Conversion

Real geotechnical data often comes as NumPy arrays (from pandas, CSV files). Seamless conversion is essential.

In [ ]:
# ─────────────────────────────────────────────────────────
# TENSOR <-> NUMPY CONVERSION
# ─────────────────────────────────────────────────────────

# NumPy -> Tensor
np_data = np.array([15.0, 18.5, 20.1, 22.3, 25.0])  # unit weights
tensor_from_np = torch.from_numpy(np_data)
print(f"NumPy -> Tensor: {tensor_from_np}")
print(f"  dtype: {tensor_from_np.dtype}  (inherits float64 from numpy)")

# WARNING: torch.from_numpy shares memory!
np_data[0] = 99.9
print(f"  After modifying numpy: tensor[0] = {tensor_from_np[0]}  (shared memory!)")
np_data[0] = 15.0  # restore

# Safe copy (no shared memory)
tensor_copy = torch.tensor(np_data)  # always copies
np_data[0] = 99.9
print(f"  torch.tensor() copy: tensor[0] = {tensor_copy[0]}  (independent copy)")
np_data[0] = 15.0

# Tensor -> NumPy
tensor_data = torch.tensor([100.0, 200.0, 350.0, 500.0])
np_from_tensor = tensor_data.numpy()
print(f"\nTensor -> NumPy: {np_from_tensor}")
print(f"  type: {type(np_from_tensor)}")

# If tensor has requires_grad=True, must detach first
grad_tensor = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
# grad_tensor.numpy()  # This would raise an error!
np_detached = grad_tensor.detach().numpy()
print(f"\nDetach + numpy: {np_detached}")

---

## Part 7: GPU Concept — Device Management

PyTorch tensors can live on CPU or GPU. GPUs are crucial for training large models on big geotechnical datasets.

In [ ]:
# ─────────────────────────────────────────────────────────
# GPU / DEVICE MANAGEMENT
# ─────────────────────────────────────────────────────────

# Check available device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create tensor on specific device
cpu_tensor = torch.tensor([1.0, 2.0, 3.0])  # default: CPU
print(f"\nCPU tensor device: {cpu_tensor.device}")

# Move tensor to device (GPU if available, otherwise stays on CPU)
device_tensor = cpu_tensor.to(device)
print(f"Device tensor:     {device_tensor.device}")

# Create directly on device
direct = torch.zeros(3, 3, device=device)
print(f"Direct on device:  {direct.device}")

# Important: tensors must be on the SAME device for operations
a = torch.tensor([1.0, 2.0]).to(device)
b = torch.tensor([3.0, 4.0]).to(device)
c = a + b  # works: both on same device
print(f"\nSame-device addition: {c}")

# Move back to CPU for plotting or numpy conversion
result_cpu = c.cpu()
print(f"Back to CPU: {result_cpu}")

print("\n--- GPU Summary ---")
print("CPU:  Universal, good for small datasets and inference")
print("GPU:  Massive parallelism, 10-100x faster for training")
print("Rule: Keep all model parameters and data on the same device")

---

## Part 8: Full Geotechnical Application — Overburden Stress Profile

Now let us put everything together: compute a complete overburden stress profile through a multi-layer soil deposit using only PyTorch tensors.

In [ ]:
# ─────────────────────────────────────────────────────────
# OVERBURDEN STRESS PROFILE — Define soil layers
# ─────────────────────────────────────────────────────────

# Define soil layers: [thickness (m), unit_weight (kN/m3)]
layer_thickness = torch.tensor([2.0, 4.0, 3.0, 6.0])   # meters
layer_gamma = torch.tensor([18.0, 16.0, 19.0, 20.0])    # kN/m3
layer_names = ['Fill', 'Soft Clay', 'Sand', 'Stiff Clay']

# Water table depth
z_wt = torch.tensor(2.0)  # meters below surface
gamma_w = torch.tensor(9.81)  # kN/m3

# Compute depth to bottom of each layer
layer_bottom = torch.cumsum(layer_thickness, dim=0)
layer_top = torch.cat([torch.tensor([0.0]), layer_bottom[:-1]])

print("Soil Profile:")
print(f"{'Layer':<12} {'Top(m)':>7} {'Bot(m)':>7} {'Thick(m)':>9} {'gamma(kN/m3)':>13}")
print("─" * 52)
for i in range(len(layer_names)):
    print(f"{layer_names[i]:<12} {layer_top[i]:7.1f} {layer_bottom[i]:7.1f} "
          f"{layer_thickness[i]:9.1f} {layer_gamma[i]:13.1f}")
print(f"\nTotal depth: {layer_bottom[-1]:.1f} m")
print(f"Water table: {z_wt:.1f} m below surface")

In [ ]:
# ─────────────────────────────────────────────────────────
# COMPUTE STRESS AT FINE DEPTH INTERVALS
# ─────────────────────────────────────────────────────────

# Create fine depth grid
total_depth = layer_bottom[-1].item()
z = torch.linspace(0, total_depth, 300)  # 300 points

# Compute total stress at each depth
sigma_v = torch.zeros_like(z)

for i in range(len(z)):
    depth = z[i]
    stress = torch.tensor(0.0)
    remaining = depth.clone()
    
    for j in range(len(layer_thickness)):
        if remaining <= 0:
            break
        h_in_layer = torch.min(remaining, layer_thickness[j])
        stress += layer_gamma[j] * h_in_layer
        remaining -= layer_thickness[j]
    
    sigma_v[i] = stress

# Pore water pressure
u = gamma_w * torch.clamp(z - z_wt, min=0.0)

# Effective stress
sigma_eff = sigma_v - u

# Print key values
print(f"{'Depth(m)':>9} {'sigma_v(kPa)':>13} {'u(kPa)':>8} {'sigma_eff(kPa)':>15}")
print("─" * 48)
for depth_val in [0, 2, 4, 6, 9, 12, 15]:
    idx = torch.argmin(torch.abs(z - depth_val))
    print(f"{z[idx]:9.1f} {sigma_v[idx]:13.1f} {u[idx]:8.1f} {sigma_eff[idx]:15.1f}")

In [ ]:
# ─────────────────────────────────────────────────────────
# VISUALIZE THE STRESS PROFILE
# ─────────────────────────────────────────────────────────

# Convert tensors to numpy for plotting
z_np = z.numpy()
sigma_v_np = sigma_v.numpy()
u_np = u.numpy()
sigma_eff_np = sigma_eff.numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 8), sharey=True)

# Total stress
axes[0].plot(sigma_v_np, z_np, 'b-', linewidth=2)
axes[0].set_xlabel('Total Stress sigma_v (kPa)', fontsize=12)
axes[0].set_ylabel('Depth (m)', fontsize=12)
axes[0].set_title('Total Vertical Stress', fontsize=13)
axes[0].invert_yaxis()
axes[0].grid(True, alpha=0.3)

# Pore water pressure
axes[1].plot(u_np, z_np, 'c-', linewidth=2)
axes[1].axhline(y=z_wt.item(), color='blue', linestyle='--', alpha=0.5, label=f'WT = {z_wt:.0f}m')
axes[1].set_xlabel('Pore Pressure u (kPa)', fontsize=12)
axes[1].set_title('Pore Water Pressure', fontsize=13)
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=10)

# Effective stress
axes[2].plot(sigma_eff_np, z_np, 'r-', linewidth=2)
axes[2].set_xlabel("Effective Stress sigma'_v (kPa)", fontsize=12)
axes[2].set_title('Effective Vertical Stress', fontsize=13)
axes[2].invert_yaxis()
axes[2].grid(True, alpha=0.3)

# Add layer boundaries
for ax in axes:
    for i, bot in enumerate(layer_bottom.numpy()):
        ax.axhline(y=bot, color='gray', linestyle=':', alpha=0.5)
        ax.text(ax.get_xlim()[1] * 0.05, (layer_top[i].item() + bot) / 2,
                layer_names[i], fontsize=8, alpha=0.6)

plt.suptitle('Overburden Stress Profile — Computed with PyTorch Tensors', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"Maximum total stress:     {sigma_v.max():.1f} kPa at {total_depth:.1f}m")
print(f"Maximum effective stress:  {sigma_eff.max():.1f} kPa")
print(f"Maximum pore pressure:     {u.max():.1f} kPa")

---

## Practice Exercises

### Exercise 1: Add a Fifth Layer
Add a `Dense Gravel` layer (gamma = 21 kN/m3, thickness = 4m) below the Stiff Clay. Recompute and plot the stress profile.

### Exercise 2: Varying Water Table
Compute effective stress profiles for water table at 0m, 2m, 5m, and 10m. Plot all four on one graph.

### Exercise 3: K0 Condition — Horizontal Stress
Compute horizontal effective stress: `sigma'_h = K0 * sigma'_v` where K0 = 0.5 for sand and K0 = 0.7 for clay. Plot both vertical and horizontal effective stress.

In [ ]:
# EXERCISE 1: Add a fifth layer
# layer_thickness = torch.tensor([2.0, 4.0, 3.0, 6.0, 4.0])
# layer_gamma = torch.tensor([18.0, 16.0, 19.0, 20.0, 21.0])
# Recompute stress profile with the new layer

# EXERCISE 2: Water table sensitivity
# wt_depths = [0.0, 2.0, 5.0, 10.0]
# for z_wt_val in wt_depths:
#     u = gamma_w * torch.clamp(z - z_wt_val, min=0.0)
#     sigma_eff = sigma_v - u
#     plt.plot(sigma_eff.numpy(), z.numpy(), label=f'WT={z_wt_val}m')

# EXERCISE 3: K0 horizontal stress
# K0_values = torch.tensor([0.5, 0.7, 0.5, 0.7])  # per layer
# Compute sigma_h_eff = K0 * sigma_eff for each depth

---

## Key Takeaways

1. **PyTorch tensors are the foundation of deep learning.** They are like NumPy arrays but with GPU support and automatic differentiation.

2. **Multiple creation methods** exist: `torch.tensor()`, `torch.zeros()`, `torch.arange()`, `torch.linspace()`, `torch.randn()`. Choose based on your data source.

3. **Element-wise and matrix operations** work naturally. Geotechnical calculations like stress profiles translate directly to tensor operations.

4. **Tensor <-> NumPy conversion** is seamless. Use `torch.from_numpy()` (shared memory) or `torch.tensor()` (copy). Use `.numpy()` to convert back.

5. **Device management** is critical. Tensors must be on the same device (CPU or GPU) for operations. Use `.to(device)` to move them.

6. **Shape manipulation** (reshape, view, unsqueeze, squeeze, stack) is essential for preparing geotechnical data for neural networks.

---

## How This Connects to ML

| Today (Tensors) | ML Application |
|-----------------|----------------|
| `torch.tensor(data)` | Store soil properties as model inputs |
| Element-wise ops | Feature engineering (stress ratios, normalized values) |
| Matrix multiply `@` | Core of every neural network layer |
| `.to(device)` | Train on GPU for 10-100x speedup |
| `.reshape()` / `.unsqueeze()` | Prepare batch inputs for DataLoader |
| `torch.from_numpy()` | Import real lab/field data into PyTorch |

**Tomorrow:** We add `requires_grad=True` to tensors, unlocking PyTorch's autograd engine for automatic backpropagation.

---

*Day 062 of 100 — We traded NumPy arrays for PyTorch tensors and computed stress like a geotechnical engineer with GPU potential!*